# 02 — Risk-Neutral NIG Dynamics and European Call Pricing

This notebook derives and validates the risk-neutral NIG characteristic function and the Fourier pricing engine. The implementation lives in `src/option_pricing.py`; this notebook is for explanation, numerical checks and interpretation.

The convention follows the course slides on the mean-correcting martingale measure and the Bakshi–Madan / Schoutens formulas. We explicitly include the dataset's continuous dividend yield.


## 1. Mathematical construction

Let the NIG increment process have cumulant exponent `kappa(z)`. The mean-correcting risk-neutral log-price is
`log(S_T) = log(S_0) + (r-q)T + X_T - T*kappa(1)`.
Therefore `phi_T(u) = exp(i*u*(log(S_0)+(r-q)T) + T*(kappa(i*u)-i*u*kappa(1)))`.
The exponential moment condition `abs(beta + 1) < alpha` is required. Notice that `mu` cancels after this correction: it is not identifiable from European option prices under this convention.


## 2. Imports and a reference market case

Import the model and pricer classes. We first use a transparent illustrative market case; the actual SPX market rows will be loaded in the next section.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from src.nig_model import NIGParameters
from src.option_pricing import MarketInputs, NIGRiskNeutralModel, EuropeanCallPricer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print(f'Project root: {PROJECT_ROOT}')


## 3. Verify the martingale correction

For a risk-neutral stock with continuous dividend yield, `E^Q[S_T] = S_0*exp((r-q)T)`. Equivalently, `phi_T(-i)` must equal this forward expectation. This is a core model validation, not an optional numerical check.


In [ ]:
parameters = NIGParameters(alpha=8.0, beta=-2.0, delta=0.25, mu=0.10)
market = MarketInputs(spot=5000.0, risk_free_rate=0.04, dividend_yield=0.015, maturity=1.0)
model = NIGRiskNeutralModel(parameters)
first_exponential_moment = model.log_price_characteristic_function(-1j, market)
risk_neutral_forward = market.spot * math.exp((market.risk_free_rate-market.dividend_yield)*market.maturity)
assert np.isclose(first_exponential_moment.real, risk_neutral_forward, rtol=1e-10)
assert np.isclose(first_exponential_moment.imag, 0.0, atol=1e-10)
print(f'phi(-i): {first_exponential_moment.real:,.6f}')
print(f'Risk-neutral forward: {risk_neutral_forward:,.6f}')


## 4. Price one European call with Fourier inversion

The course formula is `C = S0*Pi1 - K*exp(-rT)*Pi2` without dividends. With continuous dividend yield, the stock leg becomes `S0*exp(-qT)*Pi1`. The two probabilities are evaluated by numerical quadrature in the dedicated pricing class.


In [ ]:
strike = 5000.0
pricer = EuropeanCallPricer(epsabs=1e-8, epsrel=1e-8)
call_price = pricer.price_nig(strike=strike, market=market, model=model)
print(f'NIG Fourier call price: {call_price:,.6f}')
assert 0.0 <= call_price <= market.spot


## 5. Load one actual SPX quote and compare model output

Use the cleaned calibration-candidate file, not the raw workbook. This is a smoke test of the data-to-pricer interface; it is not yet a calibrated model price because the parameters above are illustrative.


In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'calibration_candidates.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Calibration candidates not found: {DATA_PATH}')
options = pd.read_csv(DATA_PATH)
quote = options.iloc[0]
quote_market = MarketInputs(
    spot=float(quote['Spot']),
    risk_free_rate=float(quote['Risk_Free_Rate_CC']),
    dividend_yield=float(quote['Dividend_Yield']),
    maturity=float(quote['Tau']),
)
quote_price = pricer.price_nig(
    strike=float(quote['Strike']), market=quote_market, model=model
)
print(f"Expiry: {quote['Expiry']} | Strike: {quote['Strike']:.2f}")
print(f"Market mid: {quote['Market_Price']:.4f} | Illustrative NIG price: {quote_price:.4f}")


## 6. Interpretation and next step

The pricing engine is now separated from calibration. Before estimating parameters, we will verify numerical stability over the actual strike/maturity sample and then calibrate only the identifiable risk-neutral shape parameters `(alpha, beta, delta)` under this mean-correcting convention. The location parameter `mu` is not an independent calibration target here because it cancels from the risk-neutral characteristic function.
